# Step 7 — Inference / submission

**Settings: `GPU T4 x2`, `Internet OFF`.**

**Inputs to attach:**
1. Competition data
2. `rsna-knee-05-train-fold0` (notebook output — the trained checkpoint)

## The one thing that matters here

The test set has no cache. This notebook decodes raw DICOMs and must apply **exactly** the same preprocessing as the cache build — same series selection, same geometric slice ordering, same percentile normalisation, same laterality canonicalisation.

A mismatch between train and test preprocessing produces no error. It just quietly destroys your score. So the decoder below is copied verbatim from `rsna-knee-04-build-cache`; if you ever change one, change both.

## Internet must be off

Submission notebooks run without network access, so `timm.create_model(pretrained=True)` would fail. We build with `pretrained=False` and load your checkpoint, which already contains every weight. That's why you never needed a separate pretrained-weights dataset.

## Runtime budget

~1,300 studies x 3 series x 24 slices = 93,600 slices. At the cache build's measured rate that's ~9 minutes to decode and ~4 minutes on GPU — call it 15 minutes against a 9-hour limit. Enormous headroom, which is exactly what makes the Efficiency track winnable later.

### 1 · Setup and model

In [ ]:
import os, gc, glob, time, math, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom, torch, torch.nn as nn
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CKPT = glob.glob("/kaggle/input/notebooks/*/*/model_fold0.pt")[0]
print("checkpoint:", CKPT)

PX, NSL, NSER = 160, 24, 3
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()

test = pd.read_csv(f"{ROOT}/test.csv")
tser = pd.read_csv(f"{ROOT}/test_series.csv")
studies = test["StudyInstanceUID"].tolist()
print(f"test studies: {len(studies):,}  |  test series: {len(tser):,}")

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        a = a.softmax(-1).unsqueeze(-1)
        return (x * a).sum(1)

class Net(nn.Module):
    def __init__(self, backbone="efficientnet_b0", nlab=12, hid=256, pretrained=False):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))

    def forward(self, x, mask):
        B, S, T = x.shape[:3]
        f = self.enc(x.reshape(B * S * T, *x.shape[3:]))
        f = f.reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        f = self.ser_pool(f, mask)
        return self.head(f)

ck = torch.load(CKPT, map_location="cpu")
model = Net(pretrained=False).cuda()
model.load_state_dict(ck["model"])
model.eval()
print(f"loaded fold {ck['fold']} | val_auc {ck['val_auc']:.4f} | gold_auc {ck['gold_auc']:.4f} | epoch {ck['epoch']}")

### 2 · Series selection — identical rule to the cache build

Sagittal, coronal, axial; within each plane prefer fluid-sensitive, then fat-suppressed.

In [ ]:
def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in tser.groupby("StudyInstanceUID")}
have = np.array([[c is not None for c in chosen.get(s, [None]*3)] for s in studies])
print(f"sagittal {have[:,0].mean():.1%} | coronal {have[:,1].mean():.1%} | axial {have[:,2].mean():.1%}")
print(f"studies with none: {(~have.any(1)).sum()}")

### 3 · Decoder — copied verbatim from the cache build

Geometric slice ordering, per-series percentile normalisation, canonical orientation, left knees mirrored. Do not modify this without changing the cache build too.

In [ ]:
def load_series(dirpath, laterality_hint=None):
    files = os.listdir(dirpath)
    if not files:
        return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas:
        return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]
        r, c = iop[:3], iop[3:]
        n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos)
    metas = [metas[i] for i in order]
    take  = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0:
            vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    lat = laterality_hint or next((m[4] for m in metas if m[4]), "")
    if str(lat).upper().startswith("L"):
        lr_axis = int(np.argmax([abs(v[0]) for v in axes]))
        vol = np.flip(vol, axis=lr_axis)

    return (vol * 255).astype(np.uint8)


def load_study(i):
    suid = studies[i]
    out  = np.zeros((NSER, NSL, PX, PX), np.uint8)
    pres = np.zeros(NSER, np.float32)
    for k, cser in enumerate(chosen.get(suid, [None]*3)):
        if cser is None: continue
        p = f"{ROOT}/test_series/{suid}/{cser}"
        if not os.path.isdir(p): continue
        try:
            v = load_series(p)
            if v is not None:
                out[k] = v; pres[k] = 1.0
        except Exception:
            pass
    return i, out, pres

print("decoder ready")

### 4 · Decode and predict, chunked

Studies are decoded in parallel on 4 CPU workers, then each chunk goes to the GPU. Chunking keeps memory flat rather than holding all 2.4 GB of pixels at once, and overlaps CPU decoding with GPU work.

Studies where nothing decodes get 0.5 across the board — no information, so no ranking claim.

In [ ]:
CHUNK, BS = 96, 6
preds = np.full((len(studies), len(LABELS)), 0.5, np.float32)

t0 = time.time()
for c0 in range(0, len(studies), CHUNK):
    idx = list(range(c0, min(c0 + CHUNK, len(studies))))

    with Pool(4) as pool:
        res = pool.map(load_study, idx)

    vols = np.stack([r[1] for r in res]).astype(np.float32) / 255.0
    mask = np.stack([r[2] for r in res])
    ids  = [r[0] for r in res]

    # 2.5D expansion: (B, S, T, 3, H, W)
    B = len(ids)
    x = np.empty((B, NSER, NSL, 3, PX, PX), np.float32)
    for b in range(B):
        for s in range(NSER):
            for j in range(NSL):
                lo, hi = max(j-1, 0), min(j+1, NSL-1)
                x[b, s, j] = np.stack([vols[b, s, lo], vols[b, s, j], vols[b, s, hi]])

    with torch.no_grad():
        for b0 in range(0, B, BS):
            xb = torch.from_numpy(x[b0:b0+BS]).cuda()
            mb = torch.from_numpy(mask[b0:b0+BS]).cuda()
            with torch.amp.autocast("cuda"):
                p = model(xb, mb)
            preds[ids[b0:b0+BS]] = p.float().sigmoid().cpu().numpy()

    del vols, x
    gc.collect()
    el = time.time() - t0
    done = c0 + len(idx)
    print(f"  {done}/{len(studies)}  {el/done:.2f}s/study  eta {(len(studies)-done)*el/done/60:.0f} min", flush=True)

print(f"\ndone in {(time.time()-t0)/60:.1f} min")

### 5 · Write submission

In [ ]:
sub = pd.DataFrame(preds, columns=LABELS)
sub.insert(0, "StudyInstanceUID", studies)
sub.to_csv("submission.csv", index=False)

print(sub.shape)
print(sub.head())
print("\nper-label prediction spread (flat columns = model gave up on that label):")
print(sub[LABELS].describe().T[["mean", "std", "min", "max"]].round(3).to_string())
print(f"\nrows left at default 0.5: {(preds == 0.5).all(1).sum()}")